### Importing Libraries

In [1]:
import math
import pandas as pd
from pulp import  *

### Loading the Dataset

In [2]:
df = pd.read_csv("fau_staffing_conductors.csv")

### Extracting Relevant Data from the Dataset

In [3]:
# Dict to store shift details
shift_details = {
    'Shift 1': {'passengers': [], 'wage': None},
    'Shift 2': {'passengers': [], 'wage': None},
    'Shift 3': {'passengers': [], 'wage': None},
    'Shift 4': {'passengers': [], 'wage': None},
}

# Fetching wages of each shift from the dataframe and storing in shift_details
wage_df = df[df['Time Windows'] == 'Wage rate per 6h shift (EUR)'].iloc[0]
for shift in shift_details:
    shift_details[shift]['wage'] = int(wage_df[shift])


# Droping the wages row
df_without_wages = df.drop(df[df['Time Windows'] == 'Wage rate per 6h shift (EUR)'].index)

# Converting Avg_Passenger_Number column values to Numeric
df_without_wages['Avg_Passenger_Number'] = pd.to_numeric(df_without_wages['Avg_Passenger_Number'], errors='coerce')

# Fetching passengers in each shift from the dataframe and storing in shift_details
for _, row in df_without_wages.iterrows():
    passengers = int(row['Avg_Passenger_Number'])
    for shift in shift_details:
        if row[shift] == 'X':
            shift_details[shift]['passengers'].append(passengers)
            break  # since only one shift per row
            
# Rename keys in shift_details from "Shift 1" to "Shift_1", etc.
shift_details = {
    shift.replace(" ", "_"): details for shift, details in shift_details.items()
}

In [164]:
print(shift_details)

{'Shift_1': {'passengers': [32, 75, 91, 82, 26, 21], 'wage': 60}, 'Shift_2': {'passengers': [25, 19, 11, 28, 46, 38], 'wage': 45}, 'Shift_3': {'passengers': [42, 34, 18, 16, 9, 5], 'wage': 45}, 'Shift_4': {'passengers': [6, 3, 2, 0, 0, 8], 'wage': 70}}


<br></br>
## Optimisation Problem
<hr>

### Decision Variables


\begin{aligned}
x_1 &= \text{Optimal staff needed for Shift 1} \\
x_2 &= \text{Optimal staff needed for Shift 2} \\
x_3 &= \text{Optimal staff needed for Shift 3} \\
x_4 &= \text{Optimal staff needed for Shift 4}
\end{aligned}


In [165]:
# Creating the variable for the optimisation problem
shifts_decision_vars = LpVariable.dicts("Fau_Transline", shift_details.keys(), lowBound=0, cat='Integer')

### Objective Function

\begin{align*}
\text{Minimize:} \quad & 60x_1 + 45x_2 + 45x_3 + 70x_4
\end{align*}

In [166]:
# Defining the Problem
prob = LpProblem("Optimal_Staff_Need", LpMinimize)

# Defining the Objective Function
prob += lpSum(shift_details[shift]['wage'] * shifts_decision_vars[shift] for shift in shift_details), "Total_Wage_Cost"

### Constraints  


$$
\text{Service Level} = \frac{1}{10} = 0.1
$$

$$
x_1 \geq \lceil \max(\text{average number of people per hour in shift 1}) \times 0.1 \rceil
$$

$$
x_2 \geq \lceil \max(\text{average number of people per hour in shift 2}) \times 0.1 \rceil
$$

$$
x_3 \geq \lceil \max(\text{average number of people per hour in shift 3}) \times 0.1 \rceil
$$

$$
x_4 \geq \lceil \max(\text{average number of people per hour in shift 4}) \times 0.1 \rceil
$$



In [167]:
# Defining the Contraints
service_level = 1/10
for shift, details in shift_details.items():
    max_passengers = max(details['passengers'])
    required_staff = math.ceil(max_passengers * service_level)
    prob += shifts_decision_vars[shift] >= required_staff, f"{shift}_Constraint"

### Solving the Problem and Displaying the Results

In [170]:
prob.solve()
print("Status:", LpStatus[prob.status])
for variable in prob.variables():
    print("The number of workers needed for", variable.name, " are ", int(variable.varValue))


Status: Optimal
The number of workers needed for Fau_Transline_Shift_1  are  10
The number of workers needed for Fau_Transline_Shift_2  are  5
The number of workers needed for Fau_Transline_Shift_3  are  5
The number of workers needed for Fau_Transline_Shift_4  are  1
